# Exploratory data analysis

Quick look at FMA-small, MusicCaps, DEAM, splits, and processed features.

In [2]:
from pathlib import Path
import json
import sys
import pandas as pd
import numpy as np
import yaml

try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
cfg = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))
print("ROOT", ROOT)
print("primary_audio", cfg["datasets"]["primary_audio"])
print("graph.type", cfg["graph"]["type"])

[ok] cell 1


## FMA-small splits & genres

In [4]:
splits = json.loads((ROOT / "data" / "splits" / "fma_small_splits.json").read_text(encoding="utf-8"))
print({k: len(splits[k]) for k in ("train", "val", "test")})
print("genres", splits["genre_to_id"])
n_npz = len(list((ROOT / "data" / "processed" / "fma_small").glob("*.npz")))
print("processed npz", n_npz)

[ok] cell 3


## MusicCaps captions

In [6]:
mc = pd.read_csv(ROOT / "data" / "raw" / "musiccaps" / "musiccaps-public.csv")
audio = ROOT / "data" / "raw" / "musiccaps" / "audio"
n_audio = len(list(audio.glob("*.wav"))) + len(list(audio.glob("*.mp3")))
print(mc.shape, "audio_files", n_audio)
display(mc[["ytid", "start_s", "end_s", "caption", "is_audioset_eval"]].head(3))

[ok] cell 5


## DEAM emotion annotations

In [8]:
from src.multilabel_data import load_deam_static_annotations

deam = load_deam_static_annotations(ROOT / "data" / "raw" / "deam")
print(deam.describe())
n_deam = len(list((ROOT / "data" / "processed" / "deam").glob("*.npz")))
print("processed deam npz", n_deam)
deam_splits = json.loads((ROOT / "data" / "splits" / "deam_splits.json").read_text(encoding="utf-8"))
print({k: len(deam_splits[k]) for k in ("train", "val", "test")})

[ok] cell 7


## Example processed features / graph sizes

In [10]:
from src.graph_builder import build_chord_transition_graph, build_segment_graph

npz = next((ROOT / "data" / "processed" / "fma_small").glob("*.npz"))
arr = np.load(npz)
print(npz.name, {k: np.asarray(arr[k]).shape for k in arr.files if hasattr(arr[k], "shape")})
cg = build_chord_transition_graph(arr["chroma"])
sg = build_segment_graph(arr["segment_vectors"], 0.7, True, True)
print("chord graph", cg.num_nodes, "nodes", cg.edge_index.size(1), "edges")
print("segment graph", sg.num_nodes, "nodes", sg.edge_index.size(1), "edges")

[ok] cell 9


## Metrics snapshot

In [12]:
metrics_path = ROOT / "results" / "metrics.json"
if metrics_path.exists():
    m = json.loads(metrics_path.read_text(encoding="utf-8"))
    print(list(m.keys()))
else:
    print("No metrics yet — run training first.")

[ok] cell 11
